# ElasticNet CV with Optional Mapper Node Selection

This notebook performs sparse modeling (ElasticNetCV) to identify ASVs associated with a morphometric trait.  
Optionally, samples can be filtered to a specific subset of Mapper nodes (include or remove mode).

**Workflow:**
1. Set user-defined parameters (population, morphometric trait, input CSV)
2. Run data loading and preprocessing
3. Optionally select Mapper nodes (pickle + cluster info file via file dialog)
4. Run ElasticNetCV across 10 L1 ratios (× 10 repeats, or LOO)
5. Inspect visualization outputs and **choose `l1_selected`**
6. Run characterization and save results


## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import pickle
import re
import os
import sys
import itertools as IT

import sklearn
import seaborn as sns
from matplotlib import pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNetCV, LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold, ParameterGrid, train_test_split
from scipy import stats
import statsmodels.api as sm

import tkinter as tk
import tkinter.filedialog
import tkinter.simpledialog as simpledialog

import dill

## 1. User-Defined Parameters
**Edit this cell before running the notebook.**

In [ ]:
# Filename of the ASV CSV file (must be in the same folder as this notebook).
INPUT_CSV = "ASV_DiffDiet.csv"

# Population to analyze.
# Options: "SF", "Pa", "Mo", "F1sp", "F1sm", "all"
pop2 = "Mo"

# Morphometric trait to use as the response variable (Y).
# Options: "StandardLength", "std_HindgutLength", "std_MidgutWidth", "std_TotalGutLength"
morph_trait = "std_HindgutLength"

# Gut region to analyze.
gut = "intestine"

# Cross-validation mode:
#   LOO = True  -> Leave-One-Out CV (1 repeat)
#   LOO = False -> 10-fold CV x 10 repeats
LOO = False

# Consistency rate threshold for selecting reliably appearing ASVs
# (fraction of repeats in which an ASV must appear to be considered consistent)
consistencyRate = 0.8

## 2. Data Loading and Preprocessing

In [ ]:
# Build CSV path relative to this notebook's location
src = os.path.dirname(os.path.abspath("__file__"))
df_all = pd.read_csv(os.path.join(src, INPUT_CSV), index_col=False, header=0)

# Filter by gut region and population
df = df_all.query('gut == "' + gut + '"')
if pop2 != 'all':
    df = df.query('pop2 == "' + pop2 + '"')

# Drop samples missing the morphometric trait or ASV data
df = df.dropna(subset=[morph_trait])
df = df.dropna(subset=["ASV_00001"])

# Set response variable Y and predictor matrix X
Y = df[morph_trait]

morp_ind = df_all.columns.get_loc('std_MidgutWidth')
X_ori = df.iloc[:, morp_ind + 1:]

# Replace exact 1 values to avoid log2(1)=0 being treated as absence
ind_1 = X_ori == 1
X_ori[ind_1.values] = 1.00001

# Remove ASVs present in only one sample (insufficient for regression)
ind_morethan1ASV = X_ori > 0
ind_morethan1ASV = ind_morethan1ASV * 1
ind_morethan1ASV = sum(ind_morethan1ASV.values)
ind_morethan1ASV = ind_morethan1ASV > 1
X_ori = X_ori.iloc[:, ind_morethan1ASV]

# Log2-transform ASV counts; replace -inf (from log2(0)) with 0
X = np.log2(X_ori)
X = X.replace([np.inf, -np.inf], 0)

print(f"Data loaded: {X.shape[0]} samples, {X.shape[1]} ASVs")
print(f"Y ({morph_trait}): mean={Y.mean():.4f}, std={Y.std():.4f}")

## 3. Optional: Mapper Node Selection

Run this cell to filter samples based on Mapper nodes.  
- A file dialog will ask for the **nodes pickle file** (from `DiffDiet_1_mapper_graph.py`).
- A second dialog will ask for a **cluster info .txt file** specifying which nodes to include or remove.


**To skip node selection and use all samples, click Cancel in the file dialog.**

In [ ]:
# --- Select Mapper nodes pickle file ---
print("Please select the Mapper nodes pickle file (or Cancel to use all samples).")
tk.Tk().withdraw()
args = {"filetypes": [("pickle", "*.pickle")],
        "title": "Select nodes pickle file. Click Cancel to use all samples."}
f_path = tkinter.filedialog.askopenfilename(**args)

if len(f_path) != 0:
    # Load the Mapper nodes
    with open(f_path, mode='br') as fi:
        nodes = pickle.load(fi)

    # --- Select cluster info file (include or remove) ---
    cluster_file = tkinter.filedialog.askopenfilenames(
        title="Select 'include' or 'remove' cluster info .txt file",
        filetypes=[("text", "*.txt")])
    clusters = list()
    file = os.path.splitext(os.path.basename(cluster_file[0]))[0]

    # Verify that the cluster file matches the selected population
    if '_' + pop2 + '_' in file:
        pop2 = pop2
    else:
        print("Error: population in cluster filename does not match pop2. Stopping.")
        sys.exit()

    # Parse cluster info file:
    # If filename contains multiple '_cuN' patterns, it is a minPath-style file
    # with node names directly; otherwise it is a tab-separated index file.
    if len(re.findall('_cu\d+', cluster_file[0])) > 1:
        cluster_info = pd.read_table(cluster_file[0], header=None)
        for i in range(len(cluster_info)):
            node_list = nodes.get(cluster_info.iloc[i, 0])
            clusters.extend(node_list)
    else:
        cluster_info = np.loadtxt(cluster_file[0], delimiter='\t', dtype='int')
        for i in range(len(cluster_info)):
            node_list = nodes.get('cube' + str(cluster_info[i, 0]) + '_cluster' + str(cluster_info[i, 1]))
            clusters.extend(node_list)

    # Remove duplicate sample indices (samples belonging to multiple nodes)
    clusters = list(set(clusters))

    # Apply filter: remove or include the selected samples
    if "remove" in cluster_file[0] or "Remove" in cluster_file[0]:
        # Remove selected samples from X and Y
        X = X.drop(df.index[clusters])
        ind_morethan1ASV = X > 0
        ind_morethan1ASV = ind_morethan1ASV * 1
        ind_morethan1ASV = sum(ind_morethan1ASV.values)
        ind_morethan1ASV = ind_morethan1ASV > 1
        X = X.iloc[:, ind_morethan1ASV]
        X_ori = X_ori.iloc[:, ind_morethan1ASV]
        Y = Y.drop(df.index[clusters])
        df = df.drop(df.index[clusters])
    else:
        # Keep only the selected samples
        X = X.iloc[clusters, :]
        ind_morethan1ASV = X > 0
        ind_morethan1ASV = ind_morethan1ASV * 1
        ind_morethan1ASV = sum(ind_morethan1ASV.values)
        ind_morethan1ASV = ind_morethan1ASV > 1
        X = X.iloc[:, ind_morethan1ASV]
        X_ori = X_ori.iloc[:, ind_morethan1ASV]
        Y = Y.iloc[clusters]
        df = df.iloc[clusters]

    print(f"After node filtering: {X.shape[0]} samples, {X.shape[1]} ASVs")
else:
    print("No pickle file selected — using all samples.")

## 4. ElasticNetCV: 10 L1 Ratios × Repeats

In [ ]:
# Define parameter grid:
#   l1_ratio: controls the mix between L1 (Lasso) and L2 (Ridge) penalties
#             1.0 = pure Lasso, 0.1 = mostly Ridge
#   repeat:   index for repeat runs (ignored for LOO)
if LOO == True:
    param_grid = {'l1_ratio': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1],
                  'repeat': [0]}
    cv = len(Y)  # Leave-One-Out
else:
    param_grid = {'l1_ratio': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1],
                  'repeat': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]}
    cv = 10

grid = ParameterGrid(param_grid)

clfs = []
MSEs = []
numASVs = []
SumNameASVs = []
repeatCountASVs = []
clfs_temp = []
MSEs_temp = []
numASVs_temp = []
SumNameASVs_temp = []

repeat_flag = param_grid['l1_ratio'][0]

for params in grid:
    print(params)

    kf = KFold(n_splits=cv, shuffle=True)
    clf = ElasticNetCV(l1_ratio=params['l1_ratio'], cv=kf, n_jobs = -1)
    clf.fit(X, Y)

    MSE = mean_squared_error(clf.predict(X), Y)
    numASV = sum((clf.coef_ != 0) * 1)
    nameASVs = list(X.columns[clf.coef_ != 0])

    # When l1_ratio changes, store accumulated results for the previous ratio
    if repeat_flag != params['l1_ratio']:
        clfs.append(clfs_temp)
        MSEs.append(MSEs_temp)
        numASVs.append(numASVs_temp)
        SumNameASVs.append(SumNameASVs_temp)
        repeatCountASVs.append(pd.DataFrame(pd.DataFrame(SumNameASVs_temp)
                               .value_counts().sort_index(), columns=['L1 rate ' + str(repeat_flag)]))
        clfs_temp = []
        MSEs_temp = []
        numASVs_temp = []
        SumNameASVs_temp = []

    repeat_flag = params['l1_ratio']
    clfs_temp.append(clf)
    MSEs_temp.append(MSE)
    numASVs_temp.append(numASV)
    SumNameASVs_temp.extend(nameASVs)

# Store the last l1_ratio group
clfs.append(clfs_temp)
MSEs.append(MSEs_temp)
numASVs.append(numASVs_temp)
SumNameASVs.extend(SumNameASVs_temp)
repeatCountASVs.append(pd.DataFrame(pd.DataFrame(SumNameASVs_temp)
                       .value_counts().sort_index(), columns=['L1 rate ' + str(repeat_flag)]))

print("ElasticNetCV complete.")

## 5. Visualization: MSE, # of ASVs, and Consistency

Inspect the plots below, then set `l1_selected` in **Section 6** accordingly.

In [ ]:
# --- Plot 1: MSE and number of ASVs across L1 ratios ---
meanMSE = np.mean(MSEs, 1)
stdMSE = np.std(MSEs, 1) / np.sqrt(len(param_grid['repeat']))
meanNumASVs = np.mean(numASVs, 1)
stdNumASVs = np.std(numASVs, 1) / np.sqrt(len(param_grid['repeat']))

fig = plt.figure()
ax1 = fig.add_subplot(111)
ax1.plot(param_grid['l1_ratio'], meanMSE, marker='o', color="dodgerblue")
ax1.errorbar(param_grid['l1_ratio'], meanMSE, yerr=stdMSE,
             marker='o', capthick=1, capsize=3, lw=1, color="dodgerblue")
ax1.set_ylabel('MSE')
ax2 = ax1.twinx()
ax2.plot(param_grid['l1_ratio'], meanNumASVs, marker='o', color="coral")
ax2.errorbar(param_grid['l1_ratio'], meanNumASVs, yerr=stdNumASVs,
             marker='o', capthick=1, capsize=3, lw=1, color="coral")
ax2.set_ylabel('# of ASVs')
plt.title('MSE (blue) and # of ASVs (orange) across L1 ratios')
plt.show()

In [ ]:
# --- Plot 2: Histogram of ASV appearance frequency across repeats, per L1 ratio ---
fig1 = plt.figure(figsize=(50, 50))
row = int(len(param_grid['l1_ratio']) // 4 + 1)
for i in list(range(len(param_grid['l1_ratio']))):
    ax1 = fig1.add_subplot(row, 4, i + 1)
    if len(repeatCountASVs[i]) > 0:
        repeatCountASVs[i].plot.bar(ax=ax1)
    ax1.legend(fontsize=40, loc='lower right')
    ax1.tick_params(labelsize=40)
plt.suptitle('Histogram of ASV appearance frequency across repeats, per L1 ratio')
plt.show()

In [ ]:
# --- Plot 3: Pairwise correlation of consistently appearing ASVs ---
# ASVs appearing in >= consistencyRate fraction of repeats are considered consistent
consisASVs = [repeatCountASVs[i].iloc[
    (repeatCountASVs[i] >= consistencyRate * len(param_grid['repeat'])).values, :]
    for i in list(range(len(param_grid['l1_ratio'])))]
consisASVsInd = [consisASVs[i].index.values.tolist()
                 for i in list(range(len(param_grid['l1_ratio'])))]
consisASVsInd1 = []

fig1 = plt.figure(figsize=(20, 15))
row = int(len(param_grid['l1_ratio']) // 3 + 1)
for i in list(range(len(param_grid['l1_ratio']))):
    ax1 = fig1.add_subplot(row, 3, i + 1)
    if len(consisASVsInd[i]) > 0:
        consisASVsInd_t = []
        consisASVsInd_t = [consisASVsInd_t + list(j) for j in consisASVsInd[i]]
        consisASVsInd_t = sum(consisASVsInd_t, [])
        consisASVsInd1.append(consisASVsInd_t)
        X_c = X.loc[:, consisASVsInd_t]
        corr = X_c.corr()
        mask = np.zeros_like(corr, dtype=bool)
        mask[np.triu_indices_from(mask)] = True
        cmap = sns.diverging_palette(250, 10, as_cmap=True)
        sns.heatmap(corr, ax=ax1, mask=mask, cmap=cmap,
                    norm=plt.Normalize(vmin=-1, vmax=1))
    else:
        consisASVsInd1.append([])
    ax1.set_title(str(param_grid['l1_ratio'][i]) +
                  ' : ' + str(len(repeatCountASVs[i])) + '->' + str(len(consisASVsInd1[i])))
plt.suptitle('Pairwise correlation of consistently appearing ASVs')
plt.show()

In [ ]:
# --- Plot 4: Scatter plot of LM fit using consistent ASVs, per L1 ratio ---
linerLegressions = []
fig1 = plt.figure(figsize=(15, 15))
row = int(len(param_grid['l1_ratio']) // 3 + 1)
for i in list(range(len(param_grid['l1_ratio']))):
    ax1 = fig1.add_subplot(row, 3, i + 1)
    if len(consisASVsInd[i]) > 0:
        X_c = X.loc[:, consisASVsInd1[i]]
        reg = LinearRegression().fit(X_c, Y)
        linerLegressions.append(reg)
        ax1.scatter(reg.predict(X_c), Y, c="blue", s=10)
        slope1, reg_intercept1, r_value1, p_value1, std_err1 = stats.linregress(reg.predict(X_c), Y)
        ax1.set_xlabel('Predicted')
        ax1.set_ylabel('True')
        ax1.plot(Y, reg_intercept1 + slope1 * Y, 'r',
                 label="R2:" + str('{:.4f}'.format(reg.score(X_c, Y))))
    else:
        linerLegressions.append([])
        ax1.set_xlabel('Predicted')
        ax1.set_ylabel('True')
    ax1.legend()
    ax1.set_title('L1=' + str(param_grid['l1_ratio'][i]) +
                  ' consistent=' + str(consistencyRate) +
                  ' : ' + str(len(repeatCountASVs[i])) + '->' + str(len(consisASVsInd1[i])))
plt.suptitle('Scatter plot of LM fit using consistently selected ASVs')
plt.show()

## 6. Select L1 Ratio

Based on the plots above, set `l1_selected` to the L1 ratio you want to use for characterization.  
Choose a value that balances low MSE with a manageable number of ASVs.

In [ ]:
# Set the L1 ratio to use for downstream characterization.
# Must be one of: 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0
l1_selected = 1.0

# Extract the selected model and its consistent ASVs
l1_ind = param_grid['l1_ratio'].index(l1_selected)
ASVs = consisASVsInd1[l1_ind]

clf = linerLegressions[l1_ind]
X_c = X.loc[:, consisASVsInd1[l1_ind]]

print(f"Selected L1 ratio: {l1_selected}")
print(f"Number of consistent ASVs: {len(ASVs)}")
print(f"ASVs: {ASVs}")

## 7. Characterization of Selected Model

In [ ]:
# Extract model coefficients and predictions
C = clf.coef_
# Standardized coefficients: scaled by ratio of predictor SD to response SD
C_standard = C * (X_c.var() / Y.values.var()) ** (0.5)
C_standard = C_standard.values
intercept = clf.intercept_
Y_pred = clf.predict(X_c)

# --- R2 reduction: contribution of each ASV to model fit ---
# For each ASV, temporarily set its coefficient to 0 and measure the drop in R2
C1 = []
C1[:] = C[:]
r2_reduce = []
for i in list(range(len(C1))):
    C1[i] = 0
    r2 = r2_score(Y, X_c @ C1 + intercept)
    r2_reduce.append(clf.score(X_c, Y) - r2)
    C1[i] = C[i]

# --- Plot: R2 reduction per ASV ---
fig2 = plt.figure(figsize=(7, 3))
plt.bar(ASVs, r2_reduce, color='maroon', width=0.4)
plt.xticks(rotation=90)
plt.title('R2 reduction per ASV: L1 ratio ' + str(l1_selected))
plt.ylabel('R2 reduction value')
plt.show()

# --- Plot: Raw and standardized coefficients ---
fig = plt.figure()
ax1 = fig.add_subplot(1, 2, 1)
ax1.bar(ASVs, C, color='r')
ax1.set_title('Coefficients: L1 ratio ' + str(l1_selected))
plt.xticks(rotation=90)
ax2 = fig.add_subplot(1, 2, 2)
ax2.bar(ASVs, C_standard, color='r')
ax2.set_title('Standardized Coefficients: L1 ratio ' + str(l1_selected))
plt.xticks(rotation=90)
plt.show()

In [ ]:
# --- Scatter plot of each ASV vs morphometric trait, colored by diet ---
diet = df['diet']
BS_ind = diet == 'BrineShrimp'
SP_ind = diet == 'Spliru'
GN_ind = diet == 'Guano'

num = len(ASVs)
row = int(num // 3 + 1)
fig1 = plt.figure(figsize=(15, 15))
for i in list(range(num)):
    # Exclude zero-abundance samples for the regression line (no-zero fit)
    df_nonZero = pd.concat([Y, X_c.iloc[:, i]], axis=1)
    df_nonZero = df_nonZero[df_nonZero[df_nonZero.columns[1]] > 0]

    ax1 = fig1.add_subplot(row, 3, i + 1)
    ax1.scatter(Y[BS_ind], X_c.iloc[:, i][BS_ind], c="orange", s=40)
    ax1.scatter(Y[SP_ind], X_c.iloc[:, i][SP_ind], c="mediumseagreen", s=40)
    ax1.scatter(Y[GN_ind], X_c.iloc[:, i][GN_ind], c="black", s=40)
    ax1.set_xlabel(Y.name)
    ax1.set_ylabel(X_c.iloc[:, i].name)
    slope2, reg_intercept2, r_value2, p_value2, std_err2 = stats.linregress(
        df_nonZero.iloc[:, 0], df_nonZero.iloc[:, 1])
    ax1.plot(Y, reg_intercept2 + slope2 * Y, 'r',
             label="R2/no zero:" + str('{:.4f}'.format(r_value2 * r_value2)))
    slope1, reg_intercept1, r_value1, p_value1, std_err1 = stats.linregress(Y, X_c.iloc[:, i])
    ax1.plot(Y, reg_intercept1 + slope1 * Y, 'b',
             label="R2/all:" + str('{:.4f}'.format(r_value1 * r_value1)))
    ax1.legend()
plt.suptitle('Scatter plot: orange=BrineShrimp, green=Spirulina, black=Guano | L1 ratio ' + str(l1_selected))
plt.show()

In [ ]:
# --- P-values for multiple regression predictors ---
# Using OLS framework to obtain t-statistics and p-values for each coefficient
XpC = sm.add_constant(X_c)
params = np.append(clf.intercept_, clf.coef_)
predictions = clf.predict(X_c)

MSE = (sum((Y - predictions) ** 2)) / (len(XpC) - len(XpC.columns))
var_b = MSE * (np.linalg.inv(np.dot(XpC.T, XpC)).diagonal())
sd_b = np.sqrt(var_b)
ts_b = params / sd_b
p_values = [2 * (1 - stats.t.cdf(np.abs(i), (len(XpC) - len(XpC.columns)))) for i in ts_b]

myDF3 = pd.DataFrame()
myDF3["Coefficients"] = np.round(params, 4)
myDF3["Standard Errors"] = np.round(sd_b, 3)
myDF3["t values"] = np.round(ts_b, 3)
myDF3["Probabilities"] = np.round(p_values, 3)

print(myDF3)

## 8. Save Results

In [ ]:
# Build output table with ASV statistics
o_ASV = X_c.columns.insert(0, 'intercept')
# Simplify ASV names by removing leading zeros (e.g. ASV_001 -> ASV_1)
o_ASV = o_ASV.str.replace('_0', '_')
o_ASV = o_ASV.str.replace('_0', '_')
o_ASV = o_ASV.str.replace('_0', '_')
o_ASV = o_ASV.str.replace('_0', '_')
o_R2_reduc = r2_reduce.copy()
o_R2_reduc.insert(0, 0)
o_C = list(C)
o_C.insert(0, clf.intercept_)
o_C_s = list(C_standard)
o_C_s.insert(0, clf.intercept_)

otu_df = pd.DataFrame(data={
    'ASV': o_ASV,
    'R2_reduction_value': o_R2_reduc,
    'Coefficients': o_C,
    'Standardized_Coefficients': o_C_s
})

# Add per-ASV simple regression statistics (all samples and non-zero samples)
otu_df['p_value'] = ''
otu_df['reg_slope'] = ''
otu_df['reg_intercept'] = ''
otu_df['reg_Rsquare'] = ''
otu_df['reg_p_value'] = ''
otu_df['no 0 reg_slope'] = ''
otu_df['no 0 reg_intercept'] = ''
otu_df['no 0 reg_Rsquare'] = ''
otu_df['no 0 reg_p_value'] = ''
otu_df.iloc[0, 5:9] = 0

num = X_c.shape[1]
for i in list(range(num)):
    df_nonZero = pd.concat([Y, X_c.iloc[:, i]], axis=1)
    df_nonZero = df_nonZero[df_nonZero[df_nonZero.columns[1]] > 0]
    otu_df.iloc[i + 1, 4] = myDF3.iloc[i + 1, 3]
    otu_df.iloc[i + 1, 5:9] = stats.linregress(Y, X_c.iloc[:, i])[0:4]
    otu_df.iloc[i + 1, 7] = stats.linregress(Y, X_c.iloc[:, i])[2] ** 2
    otu_df.iloc[i + 1, 9:13] = stats.linregress(df_nonZero.iloc[:, 0], df_nonZero.iloc[:, 1])[0:4]
    otu_df.iloc[i + 1, 11] = stats.linregress(df_nonZero.iloc[:, 0], df_nonZero.iloc[:, 1])[2] ** 2

print(otu_df)

In [ ]:
# --- Build output filename ---
if len(f_path) != 0:
    ind_t1 = f_path.rfind("/")
    cluster_file_st = ",".join(cluster_file)
    ind_t2 = cluster_file_st.rfind("/")
    pickel_f = f_path[ind_t1 + 1:-7]
    cluster_fn = cluster_file_st[ind_t2 + 1:-4]
    f_name_save = pickel_f + '_' + cluster_fn
    f_name_save = f_name_save.replace(pop2, '')
    f_name_save = f_name_save.replace(gut, '')
    f_name_save = '_' + pop2 + '_' + morph_trait + '_' + gut + '_' + f_name_save
    f_name_save = f_name_save.replace('___', '_').replace('__', '_')
    f_path_save = f_path[0:ind_t1 + 1]
else:
    f_name_save = '_' + pop2 + '_' + morph_trait + '_' + gut + '_all'
    f_path_save = src

# Avoid duplicate morph_trait in filename
if f_name_save.count(morph_trait) > 1:
    f_name_save = cluster_fn

# Prepend CV mode prefix
if LOO == True:
    f_name_save = 'L1rate_' + str(l1_selected) + '_LOO_' + f_name_save
else:
    f_name_save = 'L1rate_' + str(l1_selected) + '_rep' + str(consistencyRate) + f_name_save

# Shorten common strings for cleaner filenames
for old, new in [
    ('__', '_'), ('___', '_'), ('minPath', ''), ('KMeans', 'KM'),
    ('clusters', 'clu'), ('L1rate', 'L1'), ('node', ''),
    ('n_clus=', 'clu'), ('n_clu=', 'clu'), ('n_cube', 'n_cu'),
    ('__', '_'), ('_cl', 'cl'), ('MidgutWidth', 'MGW'),
    ('StandardLength', 'SL'), ('TotalGutLength', 'TGL'),
    ('HindgutLength', 'HGL'), ('0.8s', '0.8')
]:
    f_name_save = f_name_save.replace(old, new)

# --- Save dialog: choose output filename and location ---
from tkinter import filedialog
tk.Tk().withdraw()
filename = filedialog.asksaveasfilename(
    title='Select save location and modify filename if needed',
    initialdir=f_path_save,
    initialfile='ENCV_' + f_name_save + '_ASVs.csv'
)

# Save ASV data for selected samples
df_save = pd.concat([df.iloc[:, 0:morp_ind + 1], df[ASVs]], axis=1)
df_save.to_csv(filename)

# Save coefficient table
otu_df.to_csv(filename.replace('_ASVs.csv', '_Coeffs.csv'))

# Save full session (all variables) for later reloading
dill.dump_session(filename.replace('_ASVs.csv', '_results.pkl'))

print(f"Results saved to: {filename}")